# 🎓 SWAL Assistant Fine-Tuning - Gemma 4 E2B

**Model:** Google Gemma 4 Edge 2 Billion (E2B) - Optimized for Phone
**Size:** 3.4 GB (Q4_K_M GGUF)
**Target:** Mobile deployment (iPhone 14+, Android with Termux)

**Local Ollama:** `gemma-4-e2b-q4` (already downloaded)

**Goals:**
- SWAL Context: 0% → 90%+ (BELA, ManteniApp, Leonardo Duque, etc)
- Factuality: Knows real SWAL data
- Runs on phone! 📱

In [ ]:
# GPU Check
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes trl huggingface_hub

from google.colab import files
import os

In [ ]:
# Upload training data from local or Google Drive
print("Upload xavier_training.jsonl:")
uploaded = files.upload()

# Or mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/training_data/xavier_training.jsonl .

In [ ]:
# Load and inspect training data
import json
from collections import Counter

with open("xavier_training.jsonl", 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

print(f"Total examples: {len(data)}")

categories = Counter([d.get('category', 'unknown') for d in data])
print("\nCategories:")
for cat, count in categories.most_common():
    print(f"  {cat}: {count}")

print("\nSample:")
print(f"  Q: {data[0]['instruction']}")
print(f"  A: {data[0]['response'][:100]}...")

In [ ]:
# Format for Gemma 4 instruction tuning
from datasets import Dataset

def format_gemma4(example):
    """Format for Gemma 4's instruction tuning format with thinking support."""
    return {
        "text": f"<start_of_turn>user\n{example['instruction']}<end_of_turn>\n<start_of_turn>model\n{exampl

## 📁 Option 1: Fine-tune from HF (Recommended for Colab)

Download Gemma 4 E2B from HuggingFace and fine-tune with QLoRA.

In [ ]:
# Option 1: Fine-tune from HuggingFace (google/gemma-4-E2B)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "google/gemma-4-E2B"  # Or bartowski/gemma-4-E2B-GGUF for lower memory

# QLoRA config - 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Loading Gemma 4 E2B with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)
print("Model loaded!")

In [ ]:
# Configure LoRA for Gemma 4
lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Training config
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./swal-gemma4-e2b-v1",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optimization="adamw_torch",
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_seq_length=512,
    dataset_text_field="text",
)

In [ ]:
# Start training!
print("=" * 50)
print("Starting Gemma 4 E2B Fine-Tuning")
print("=" * 50)
print(f"Model: google/gemma-4-E2B")
print(f"Training examples: {len(dataset)}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Estimated time: 1-2 hours on A100")

model.train()
trainer.train()

In [ ]:
# Save model
model.save_pretrained("./swal-gemma4-e2b-v1")
tokenizer.save_pretrained("./swal-gemma4-e2b-v1")
print("Model saved to ./swal-gemma4-e2b-v1")

In [ ]:
# Test the fine-tuned model
from transformers import pipeline

test_pipe = pipeline(
    "text-generation",
    model="./swal-gemma4-e2b-v1",
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

test_queries = [
    "Quién es BELA y qué proyectos tiene SWAL?",
    "Cuál es el precio de ManteniApp?",
    "Quién es Leonardo Duque?",
]

for query in test_queries:
    print(f"\nQ: {query}")
    result = test_pipe(
        f"<start_of_turn>user\n{query}<end_of_turn>\n<start_of_turn>model\n",
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
    )
    response = result[0]['generated_text'].split('<start_of_turn>model\n')[1]
    print(f"A: {response}")

## 📁 Option 2: Convert to GGUF for Ollama/Termux (Phone)

After fine-tuning, convert to GGUF format for local Ollama or Termux deployment.

In [ ]:
# Convert fine-tuned model to GGUF for Ollama
!pip install -q llama-cpp-python

# Option A: Upload to HuggingFace then download GGUF
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="./swal-gemma4-e2b-v1",
#     repo_id="your-username/swal-gemma4-e2b",
#     repo_type="model"
# )

# Option B: Use conversion script
# !python llama.cpp/convert-hf-to-gguf.py ./swal-gemma4-e2b-v1 --outfile ./swal-gemma4-e2b.gguf

print("GGUF conversion ready!")

## 📱 Mobile Deployment Guide

### Termux (Android) - Recommended
```bash
# Install Termux from F-Droid (not Google Play)
pkg update && pkg install python git cmake

# Clone and build llama.cpp
git clone https://github.com/ggerganov/llama.cpp.git
cd llama.cpp
cmake . && make

# Download fine-tuned model from HuggingFace
# Upload to Drive, download to phone

# Run inference
./main -m ./swal-gemma4-e2b-q4.gguf -n 512 -t 4 --temp 0.7
```

### iOS (MLC Chat)
- Use MLC Chat app from App Store
- Import model from HuggingFace
- iPhone 14+ can run E2B at interactive speeds

### Ollama (PC/Mac)
```bash
# Copy GGUF to Ollama models directory
cp swal-gemma4-e2b-q4.gguf ~/.ollama/models/

# Create Modelfile
echo 'FROM ./swal-gemma4-e2b-q4.gguf' > Modelfile
ollama create swal-assistant -f Modelfile

# Run
ollama run swal-assistant
```

In [ ]:
# Download model for local use (optional)
import zipfile
import os

def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.add(file_path, arcname)

print("Zipping fine-tuned model...")
zip_folder("./swal-gemma4-e2b-v1", "./swal-gemma4-e2b-v1.zip")
print(f"Model size: {os.path.getsize('./swal-gemma4-e2b-v1.zip') / 1e9:.2f} GB")

files.download("./swal-gemma4-e2b-v1.zip")